# Opik Dashboards — Python SDK

This notebook walks through the high-level **dashboard** API on the `opik.Opik` client:

- `create_dashboard` / `get_dashboard` / `find_dashboards` / `delete_dashboard`
- The `Dashboard` read-modify-write helpers (`add_widget`, `update_widget`, `remove_widget`, `rename`, ...)
- Typed widget builders under `opik.dashboard`

It also highlights the **two metric-ID namespaces** that are easy to mix up:

| Widget | Field | Namespace | Example |
| --- | --- | --- | --- |
| `project_stats_card` | `metric` | lowercase-dotted | `trace_count`, `duration.p50` |
| `project_metrics` | `metric_type` | ALL-CAPS | `TRACE_COUNT`, `DURATION` |

## Setup

In [ ]:
%pip install opik --quiet

In [ ]:
import opik
from opik import dashboard

# Configure once. The client reads OPIK_API_KEY / OPIK_URL_OVERRIDE / OPIK_WORKSPACE
# from the environment if set. For a local backend, e.g.:
#   export OPIK_URL_OVERRIDE=http://localhost:5173/api
#   export OPIK_WORKSPACE=default
client = opik.Opik()

# A project that exists in your workspace — the project-scoped widgets point at it.
PROJECT_NAME = "Default Project"

## Create a dashboard

A new dashboard starts with a single empty `Overview` section. We grab its id to add widgets to it.

In [ ]:
dash = client.create_dashboard(
    name="SDK demo dashboard",
    type=dashboard.DashboardType.MULTI_PROJECT,
    description="Created from the Python SDK",
)
section_id = dash.sections[0].id
print(dash.id, dash.type, dash.scope)
section_id

## Resolve a project id

The `project_stats_card` and `project_metrics` widgets reference a project by id.

In [ ]:
project = client.rest_client.projects.retrieve_project(name=PROJECT_NAME)
project.id

## Add widgets

We add three widgets, exercising both metric namespaces and both construction styles
(typed builders and a raw dict). Each `add_widget` auto-places the widget on the grid.

In [ ]:
# 1. Single-metric stats card — lowercase-dotted metric namespace.
dash.add_widget(
    section_id,
    dashboard.DashboardWidget(
        type=dashboard.WidgetType.PROJECT_STATS_CARD,
        title="Total traces",
        config=dashboard.ProjectStatsCardConfig(
            project_id=project.id,
            source=dashboard.TraceDataType.TRACES,
            metric=dashboard.StatsCardMetric.TRACE_COUNT,
        ),
    ),
)

# 2. Time-series chart with a breakdown — ALL-CAPS metric_type namespace.
dash.add_widget(
    section_id,
    dashboard.DashboardWidget(
        type=dashboard.WidgetType.PROJECT_METRICS,
        title="Duration over time, by model",
        config=dashboard.ProjectMetricsConfig(
            project_id=project.id,
            metric_type=dashboard.ProjectMetricType.DURATION,
            chart_type=dashboard.ChartType.LINE,
            breakdown=dashboard.BreakdownConfig(field=dashboard.BreakdownField.MODEL),
        ),
    ),
)

# 3. Markdown widget built from a raw dict (power-user / forward-compatible path).
markdown_id = dash.add_widget(
    section_id,
    {
        "type": dashboard.WidgetType.TEXT_MARKDOWN.value,
        "title": "Notes",
        "config": {"content": "# Demo\nBuilt with the Opik Python SDK."},
    },
)
print(f"{len(dash.sections[0].widgets)} widgets")

## Read-modify-write

Mutators fetch nothing extra — they edit the local config and PATCH the whole document back.

In [ ]:
dash.update_widget(markdown_id, config={"content": "# Demo (updated)"})
dash.rename("SDK demo dashboard (renamed)")
dash.name

## Fetch and find

In [ ]:
fetched = client.get_dashboard(dash.id)
print(fetched.name)

found = client.find_dashboards(name="SDK demo dashboard", max_results=10)
print(f"found {len(found)} dashboard(s)")

## Inspect the persisted config

The config round-trips through the backend as camelCase JSON. Note the two metric namespaces
and that every widget has a matching layout item.

In [ ]:
import json

config = fetched.config
print("version:", config["version"])
for w in config["sections"][0]["widgets"]:
    print(w["type"], "->", json.dumps(w["config"]))
print(
    "layout:",
    [
        (i["i"][:8], i["x"], i["y"], i["w"], i["h"])
        for i in config["sections"][0]["layout"]
    ],
)

## Clean up

In [ ]:
dash.delete()
print("deleted")